# AM 11B Lab Activity 2: Profit Maximization in Two Variables

**Week 2 — built on: Optimization in Several Variables (critical points, second derivative test)**

**Authors:** Julie Simons, with Claude (Sonnet 5, Anthropic), via Claude Code — September 2026

*With special thanks to Mariana Bessa and prior AM 11B students, whose earlier notebooks inspired this material, and to Eric Van Dusen and the [Data 88E](https://github.com/data-88e) course for inspiration.*

![Stacked coins with small plants growing from them, representing growth](https://thumb.wikimedia.org/wikipedia/commons/thumb/0/04/Money_saving_growth.jpg/960px-Money_saving_growth.jpg)

*Photo: nattanan, CC0, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Money_saving_growth.jpg)*

<!-- canvas-group-instructions -->
## Group Submission Instructions

**Joining your lab group on Canvas**

1. Agree with your partners on a group number (e.g., "Lab Group 7").
2. In our Canvas course, click **People**, then the **Groups** tab.
3. Find your group and click **Join**. **Every member must do this themselves.** You can't add your partners.
4. To check, click the group name and make sure all your partners are listed. If you joined the wrong group, click **Leave** and join the right one.
5. When you submit, **only one person uploads the notebook**. It counts for everyone in the group.

Please join your group before submitting your first lab. If you submit before joining, only you will get credit.

Please write the name(s) of your group (or yourself) below by double-clicking on this text.

**NAME(S):** *Replace this text with your name(s)*

## Learning Goals

By the end of this lab you should be able to:
1. Set up a two-input profit function for a firm.
2. Find critical points by solving a system of first-order conditions ($\partial \pi/\partial L = 0$, $\partial \pi/\partial K = 0$).
3. Apply the second derivative test to confirm a critical point is a maximum.
4. Visualize the profit surface and see the maximum as the peak of a "hill."
5. Recognize that a critical point isn't automatically a maximum — see what a **saddle point** looks like, and why the second derivative test matters.

### Why economists care
Firms choose input levels to maximize profit. The math is exactly last week's optimization toolkit — critical points and the second derivative test — applied to a concrete economic function. This lab is a direct rehearsal for the kind of problem you'll see on quizzes and in the standards.

## Setup

Run the cell below to load the tools we'll use today.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sympy as sp
import random

sp.init_printing()
plt.rcParams['figure.figsize'] = (9, 6)
%matplotlib inline

def checkpoint(is_correct, hint=""):
    if is_correct:
        emoji = random.choice(["✅", "🚀", "⭐"])
        msg = random.choice([
            "Great job! You nailed it.",
            "You did it!",
            "Nailed it — nice work.",
            "Yes! That's exactly right.",
            "Correct! You're crushing this.",
        ])
        print(f"{emoji} {msg}")
    else:
        print("❌ Try again!" + (f" {hint}" if hint else ""))

print("Setup complete!")

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

## The Scenario

A firm produces output using labor $L$ and capital $K$ according to

$$Q(L,K) = 10 L^{0.4} K^{0.3}$$

It sells output at price $p = 6$ per unit, pays wage $w = 5$ per unit of labor, and pays rental rate $r = 3$ per unit of capital. Its profit function is:

$$\pi(L,K) = p\,Q(L,K) - wL - rK = 6(10 L^{0.4}K^{0.3}) - 5L - 3K$$

### Step 1: Set up the profit function and first-order conditions — try it by hand first

**Before running any code**, use the power rule (fractional exponents work exactly the same way — bring the exponent down, subtract one) to find $\partial \pi/\partial L$ and $\partial \pi/\partial K$ by hand, starting from

$$\pi(L,K) = 60 L^{0.4}K^{0.3} - 5L - 3K$$

You don't need to evaluate these anywhere or solve anything yet — just get the two formulas on paper. Write them below, then run the cell to check.

</div>

*Replace this text with your hand-derived formulas for $\partial \pi/\partial L$ and $\partial \pi/\partial K$.*

Run the cell below to define $\pi(L,K)$ symbolically and compute $\partial \pi/\partial L$ and $\partial \pi/\partial K$ with `sympy`, and compare to what you found by hand.

In [ ]:
L, K = sp.symbols('L K', positive=True)
p, w, r = 6, 5, 3
Q = 10 * L**sp.Rational(4,10) * K**sp.Rational(3,10)
pi = p*Q - w*L - r*K

pi_L = sp.diff(pi, L)
pi_K = sp.diff(pi, K)

print("pi(L,K)   =", pi)
print("d(pi)/dL  =", pi_L)
print("d(pi)/dK  =", pi_K)

expected_pi_L = 24*K**sp.Rational(3,10)/L**sp.Rational(3,5) - 5
expected_pi_K = 18*L**sp.Rational(2,5)/K**sp.Rational(7,10) - 3
checkpoint(
    sp.simplify(pi_L - expected_pi_L) == 0 and sp.simplify(pi_K - expected_pi_K) == 0,
    hint="Recheck your power-rule algebra against the printed pi_L and pi_K above."
)

### Step 2: Solve the system of first-order conditions

At a critical point, both partial derivatives equal zero simultaneously. This system isn't friendly to solve by hand (fractional exponents!), so we'll let `sympy` solve it numerically with `nsolve`, starting from a guess.

In [ ]:
critical_pt = sp.nsolve([pi_L, pi_K], [L, K], [10, 10])
Lstar, Kstar = float(critical_pt[0]), float(critical_pt[1])
pi_star = float(pi.subs({L: Lstar, K: Kstar}))

print(f"Critical point: L* = {Lstar:.3f}, K* = {Kstar:.3f}")
print(f"Profit at critical point: pi* = {pi_star:.3f}")

### Step 3: Confirm it's a maximum with the second derivative test

Recall the test: compute $\pi_{LL}$, $\pi_{KK}$, $\pi_{LK}$, then the discriminant $D = \pi_{LL}\pi_{KK} - \pi_{LK}^2$.

- If $D > 0$ and $\pi_{LL} < 0$: **local maximum**
- If $D > 0$ and $\pi_{LL} > 0$: local minimum
- If $D < 0$: saddle point
- If $D = 0$: test is inconclusive

In [ ]:
pi_LL = sp.diff(pi, L, 2)
pi_KK = sp.diff(pi, K, 2)
pi_LK = sp.diff(pi, L, K)

D_val    = float((pi_LL*pi_KK - pi_LK**2).subs({L: Lstar, K: Kstar}))
pi_LL_val = float(pi_LL.subs({L: Lstar, K: Kstar}))

print(f"D  = {D_val:.6f}")
print(f"pi_LL = {pi_LL_val:.6f}")

if D_val > 0 and pi_LL_val < 0:
    print("=> This critical point IS a local maximum. ✓")
elif D_val > 0 and pi_LL_val > 0:
    print("=> This critical point is a local MINIMUM.")
elif D_val < 0:
    print("=> This critical point is a SADDLE POINT.")
else:
    print("=> Test inconclusive.")

checkpoint(D_val > 0 and pi_LL_val < 0, hint="This should classify as a local maximum -- recheck Lstar, Kstar above.")

### Question 2.1: Reading the Result

In your own words, what does it mean economically that $\partial \pi/\partial L = 0$ and $\partial \pi/\partial K = 0$ *simultaneously* at the optimum, rather than just one of them being zero?

*Replace this text with your answer.*

## Visualizing the Profit Surface

Run the cell below to see profit plotted as a 3D surface over labor and capital — it should look like a rounded hill, rising from the edges of the plot up to a single peak. A black dot marks that peak: the maximum you just found.

In [ ]:
L_vals = np.linspace(50, 450, 40)
K_vals = np.linspace(50, 500, 40)
L_grid, K_grid = np.meshgrid(L_vals, K_vals)
pi_grid = p*10*L_grid**0.4*K_grid**0.3 - w*L_grid - r*K_grid

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(L_grid, K_grid, pi_grid, cmap='plasma', alpha=0.85)
ax.scatter([Lstar], [Kstar], [pi_star], color='black', s=80, label='Maximum')
ax.set_xlabel('Labor (L)')
ax.set_ylabel('Capital (K)')
ax.set_zlabel('Profit')
ax.set_title('Profit Surface with Maximum Marked')
ax.legend()
plt.show()

<div style="background-color:#e7f1fc; border-left:6px solid #4a90d9; border-radius:4px; padding:4px 18px 4px 18px; color:#0b3d78; margin:12px 0;">

## Your Turn

The wage rate $w$ has a big influence on how much labor a profit-maximizing firm hires. Below, change `w_new` to a **higher** wage (say, 8 or 10) and re-solve for the new optimum. The code will automatically re-solve the system and re-check the second derivative test for you — no need to redo any algebra by hand.

Edit the cell marked `# EDIT HERE`, then run it.

</div>

In [ ]:
w_new = None   # EDIT HERE -- try a wage higher than the original w=5
assert w_new is not None, "Set w_new to a number before running this cell!"

pi_new = p*Q - w_new*L - r*K
pi_L_new = sp.diff(pi_new, L)
pi_K_new = sp.diff(pi_new, K)

critical_pt_new = sp.nsolve([pi_L_new, pi_K_new], [L, K], [10, 10])
Lstar_new, Kstar_new = float(critical_pt_new[0]), float(critical_pt_new[1])
pi_star_new = float(pi_new.subs({L: Lstar_new, K: Kstar_new}))

pi_LL_new = sp.diff(pi_new, L, 2)
pi_KK_new = sp.diff(pi_new, K, 2)
pi_LK_new = sp.diff(pi_new, L, K)
D_new = float((pi_LL_new*pi_KK_new - pi_LK_new**2).subs({L: Lstar_new, K: Kstar_new}))
pi_LL_new_val = float(pi_LL_new.subs({L: Lstar_new, K: Kstar_new}))

print(f"New wage: w = {w_new}")
print(f"New optimum: L* = {Lstar_new:.3f}, K* = {Kstar_new:.3f}, profit* = {pi_star_new:.3f}")
print(f"Second derivative test: D = {D_new:.6f}, pi_LL = {pi_LL_new_val:.6f}",
      "-> still a maximum ✓" if (D_new > 0 and pi_LL_new_val < 0) else "-> NOT a maximum, check your inputs")

print(f"\nCompared to the original (w=5): L* changed from {Lstar:.2f} to {Lstar_new:.2f}")

checkpoint(
    w_new > 5 and D_new > 0 and pi_LL_new_val < 0 and Lstar_new < Lstar,
    hint="Make sure w_new is actually higher than 5 -- a higher wage should still give a maximum, with less labor hired."
)

### Question 2.2: Economic Intuition

Based on the output above, what happened to the optimal amount of labor $L^*$ when the wage went up? Does this match your intuition about how a cost-minimizing, profit-maximizing firm should respond to a more expensive input? Briefly explain.

*Replace this text with your answer.*

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

## A Cautionary Tale: When the Critical Point Is NOT a Maximum

So far, every time we solved $\partial \pi/\partial L = 0$ and $\partial \pi/\partial K = 0$, we found a maximum. That might make it tempting to assume "solve the FOCs, declare victory." **Don't.** The second derivative test exists precisely because a critical point can also be a **minimum** or a **saddle point** — and you can't tell which just by finding where both partial derivatives are zero.

Here's a deliberately different example. A firm operates in two independent markets, $x$ and $y$, with combined profit:

$$\pi(x,y) = 100x - x^2 \;+\; y^2 - 40y$$

Market $x$ behaves normally (diminishing returns — profit is concave in $x$). But suppose market $y$ has strong **increasing returns to scale** (e.g. network effects, where growth accelerates rather than tapers off) — the $+y^2$ term makes profit *convex* in $y$ rather than concave.

**This one, unlike the Cobb-Douglas example above, is small enough to fully solve by hand** — no fractional exponents, no `nsolve` needed. **Before running any code**, grab a pencil and paper:

1. Find $\partial \pi/\partial x$ and $\partial \pi/\partial y$, set each to zero, and solve — these are just simple linear equations.
2. Compute $\pi_{xx}$, $\pi_{yy}$, $\pi_{xy}$, the discriminant $D = \pi_{xx}\pi_{yy} - \pi_{xy}^2$, and use the classification rules from Step 3 above to say whether this critical point is a max, a min, or a saddle point.

Write your critical point, your value of $D$, and your classification below.

</div>

*Replace this text with your hand-computed critical point $(x^*, y^*)$, the value of $D$, and your classification (max / min / saddle).*

Now run the cell below to check your work.

In [ ]:
x_, y_ = sp.symbols('x y')
pi_xy_example = 100*x_ - x_**2 + y_**2 - 40*y_

pix = sp.diff(pi_xy_example, x_)
piy = sp.diff(pi_xy_example, y_)
solution = sp.solve([pix, piy], [x_, y_])
xstar2, ystar2 = solution[x_], solution[y_]
pi_at_crit = pi_xy_example.subs({x_: xstar2, y_: ystar2})

print(f"Critical point: x* = {xstar2}, y* = {ystar2}")
print(f"Profit at this critical point: {pi_at_crit}")

pixx = sp.diff(pi_xy_example, x_, 2)
piyy = sp.diff(pi_xy_example, y_, 2)
pixy = sp.diff(pi_xy_example, x_, y_)
D_example = pixx*piyy - pixy**2

print(f"\npi_xx = {pixx}, pi_yy = {piyy}, pi_xy = {pixy}")
print(f"D = {D_example}")

if D_example > 0 and pixx < 0:
    print("=> local maximum")
elif D_example > 0 and pixx > 0:
    print("=> local minimum")
elif D_example < 0:
    print("=> SADDLE POINT — this critical point is NOT an optimum at all!")
else:
    print("=> test inconclusive")

checkpoint(
    xstar2 == 50 and ystar2 == 20 and D_example == -4,
    hint="You should get x*=50, y*=20, and D=-4 -- recheck your FOCs and second derivatives."
)

Run the cell below to see what happens to profit if you move *away* from this critical point along each direction separately, holding the other input fixed.

In [ ]:
print("Moving along x (holding y = y*):")
for xx in [0, 25, 50, 75, 100]:
    val = float(pi_xy_example.subs({x_: xx, y_: ystar2}))
    marker = "  <-- critical point" if xx == xstar2 else ""
    print(f"  x={xx:>4}: profit = {val:>8.1f}{marker}")

print("\nMoving along y (holding x = x*):")
for yy in [0, 10, 20, 30, 40, 60]:
    val = float(pi_xy_example.subs({x_: xstar2, y_: yy}))
    marker = "  <-- critical point" if yy == ystar2 else ""
    print(f"  y={yy:>4}: profit = {val:>8.1f}{marker}")

Notice: along the $x$-direction, profit peaks right at $x^*=50$, just like before. But along the $y$-direction, profit at $y^*=20$ is actually the *lowest* point nearby — moving $y$ in **either** direction increases profit! That's the signature of a saddle point: a max along one direction and a min along another, at the very same point.

In [ ]:
x_vals3 = np.linspace(0, 100, 40)
y_vals3 = np.linspace(-20, 60, 40)
x_grid3, y_grid3 = np.meshgrid(x_vals3, y_vals3)
pi_grid3 = 100*x_grid3 - x_grid3**2 + y_grid3**2 - 40*y_grid3

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(x_grid3, y_grid3, pi_grid3, cmap='coolwarm', alpha=0.85)
ax.scatter([float(xstar2)], [float(ystar2)], [float(pi_at_crit)], color='black', s=80, label='Critical point (saddle)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('Profit')
ax.set_title('A Saddle-Shaped Profit Surface')
ax.legend()
plt.show()

### Question 2.3: Reading a Saddle Point

Look at the surface above — it curves *down* in one direction and *up* in the other, like a horse saddle (or a Pringle chip). Answer briefly:

**a)** Why would a firm operating at $(x,y)=(50,20)$ actually want to move away from this critical point, rather than stay there? (Hint: which direction would you move in, and why?)

**b)** In practice, if profit truly keeps increasing as $y$ grows without bound (increasing returns to scale, no diminishing marginal effect), what does that suggest about where the *real* optimum for $y$ would be? (Hint: is it necessarily somewhere with $\partial \pi/\partial y = 0$?)

*Replace this text with your answer.*

## Reflection

**In 1-2 sentences:** The second derivative test required *three* pieces of information ($\pi_{LL}$, $\pi_{KK}$, $\pi_{LK}$) instead of just checking whether $\pi_{LL} < 0$. Why isn't checking $\pi_{LL} < 0$ alone enough to guarantee a maximum in two variables, the way it would be in one variable?

*Replace this text with your answer.*

---

## Submission

1. Make sure you've run all cells in order (**Runtime/Kernel → Restart and run all** is a good final check).
2. Answer the short reflection question(s) above by editing the markdown cells.
3. Go to **File → Download** to save the `.ipynb` file to your computer.
4. Upload the file to Canvas under this Lab Activity.

Only one person per group needs to submit if you worked together — just make sure everyone's name is listed at the top.

**This lab is designed to take 30–60 minutes.** If you're stuck for more than a few minutes on any one cell, that's your cue to ask a neighbor, a TA, or post on Ed Discussion — don't burn your whole lab period on one line of code.
